<a href="https://colab.research.google.com/github/grmntfrancis0/earthengine-community/blob/main/LST_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade xee

In [ ]:
!pip install -U geemap

In [ ]:
import ee

In [ ]:
ee.Authenticate()
ee.Initialize(project = "ee-grmntfrancis0",
                opt_url='https://earthengine-highvolume.googleapis.com'

)

In [ ]:
import geemap

In [ ]:
map = geemap.Map()
map

In [ ]:
roi = map.draw_last_feature.geometry()
roi

In [ ]:
temp = (ee.ImageCollection("NASA/VIIRS/002/VNP21A1D")
.filterDate("2024-01-01","2025-01-31")
.filterBounds(roi)
.select("LST_1KM")

        )

temp

In [ ]:
ndvi = (
    ee.ImageCollection("NASA/VIIRS/002/VNP13A1")
    .filterDate("2024-01-01","2024-12-30")
    .filterBounds(roi)
    .select("NDVI")
)
ndvi

In [ ]:
import xarray as xr

In [ ]:
ds_temp = xr.open_dataset(
   temp,
   engine = "ee",
   crs = "EPSG:4326",
   geometry = roi,
   scale = 0.01
)
ds_temp

In [ ]:
ds_ndvi = xr.open_dataset(
    ndvi,
    engine = "ee",
    crs = "EPSG:4326",
    geometry = roi,
    scale = 0.005
)
ds_ndvi

In [ ]:
ds_temp = ds_temp.sortby("time") * 1

In [ ]:
ds_ndvi = ds_ndvi.sortby("time") * 1

In [ ]:
ds_temp_monthly = ds_temp.resample(time = "M").mean("time")
ds_ndvi_monthly = ds_ndvi.resample(time = "M").median("time")

In [ ]:
ds_temp_monthly

In [ ]:
ds_ndvi_monthly

In [ ]:
ds_ndvi_like_temp = ds_ndvi_monthly.interp_like(ds_temp_monthly)

In [ ]:
ds_temp_monthly

In [ ]:
ds_ndvi_monthly

In [ ]:
ds_ndvi_like_temp

In [ ]:
merge = xr.merge([ds_temp_monthly,ds_ndvi_like_temp])

In [ ]:
merge

In [ ]:
!pip install netCDF4

In [ ]:
import netCDF4 as nc

In [ ]:
merge.to_netcdf("merge_ds.nc")

In [ ]:
merge.LST_1KM.plot(
    x = "lon", y = "lat", col = "time", col_wrap = 6, cmap = "RdBu", robust = True

    )

In [ ]:
merge.NDVI.plot(
    x = 'lon', y = 'lat', col = 'time', col_wrap = 6, robust = True, cmap = 'RdBu_r'
)
